Лабораторная работа №2

Обработка пропусков, кодирование категориальных признаков и масштабирование данных

## **Лабораторная работа №2 — Отчет**

### **Цель работы**

Изучение способов предварительной обработки данных:

* выявление и замена пропусков
* кодирование категориальных признаков
* масштабирование числовых признаков

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### **1) Краткое описание датасета**

Датасет **Titanic** содержит данные о пассажирах рейса «Титаник», включая:

* возраст, пол, класс, стоимость билета
* цель: предсказать останется пассажир в живых (Survived) или нет


### Предварительный анализ
пропуски и типы признаков

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [5]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

### Обработка пропусков

три метода заполнения пропусков

In [8]:
# Удаление строк с пропусками

df_drop = df.dropna()
df_drop.shape

(183, 12)

In [9]:
# Заполнение средним/модой

df_fill = df.copy()

df_fill["Age"] = df_fill["Age"].fillna(df_fill["Age"].mean())
df_fill["Embarked"] = df_fill["Embarked"].fillna(df_fill["Embarked"].mode()[0])
df_fill["Cabin"] = df_fill["Cabin"].fillna("Unknown")

df_fill.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

In [ ]:
# Предсказательная имputation (пример с медианой по группам), но можно и knn

df_group = df.copy()
df_group["Age"] = df_group.groupby("Pclass")["Age"].transform(lambda x: x.fillna(x.median()))
df_group["Cabin"] = df_group["Cabin"].fillna("Unknown")
df_group["Embarked"] = df_group["Embarked"].fillna(df_group["Embarked"].mode()[0])
df_group.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

### **2) Обработка пропусков**

Показаны:

* удаление строк с пропусками
* заполнение средним/модой
* заполнение по группам



### Кодирование категориальных признаков

In [ ]:
# One-Hot Encoding (разреженное бинарное кодирование)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

categorical_cols = ["Sex", "Embarked", "Cabin"]
encoded = ohe.fit_transform(df_fill[categorical_cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(categorical_cols))
encoded_df.head()

,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Cabin_A10,Cabin_A14,Cabin_A16,Cabin_A19,Cabin_A20,...,Cabin_F E69,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Cabin_Unknown
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [17]:
# Label Encoding (простой номер категории)
from sklearn.preprocessing import LabelEncoder

le_df = df_fill.copy()
le = LabelEncoder()

for col in ["Sex","Embarked","Cabin"]:
    le_df[col] = le.fit_transform(le_df[col].astype(str))

le_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,147,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,81,0
2,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,147,2
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,55,2
4,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,147,2


In [18]:
# Mean‑Target Encoding
target_mean = df_fill.groupby("Embarked")["Survived"].mean()
df_mean = df_fill.copy()
df_mean["Embarked_encoded"] = df_mean["Embarked"].map(target_mean)

df_mean[["Embarked","Embarked_encoded"]].head()

,Embarked,Embarked_encoded
0,S,0.339009
1,C,0.553571
2,S,0.339009
3,S,0.339009
4,S,0.339009


### **3) Кодирование категориальных признаков**

Показаны 3 подхода:

* One-Hot Encoding
* Label Encoding
* Mean‑Target Encoding


### Масштабирование данных

Мы покажем масштабирование для числовых признаков:

In [ ]:
scaler = StandardScaler() # z-нормализация
num_cols = ["Age", "Fare"]

scaled = scaler.fit_transform(df_fill[num_cols])
scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in num_cols])

scaled_df.head()

,Age_scaled,Fare_scaled
0,-0.592481,-0.502445
1,0.638789,0.786845
2,-0.284663,-0.488854
3,0.407926,0.420730
4,0.407926,-0.486337



### **4) Масштабирование числовых признаков**

Использован StandardScaler. Для z преобразования


### Соберём комплексный препроцессинг в Pipeline

ColumnTransformer + Pipeline

In [20]:
# числовые и категориальные столбцы
num_features = ["Age","Fare"]
cat_features = ["Sex","Embarked","Cabin"]

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
])

### Итоговая обработанная таблица

In [23]:
processed = preprocessor.fit_transform(df_fill)
print(processed.shape)

(891, 155)





### **5) Выводы**

* Пропуски влияют на модель и требуют обработки
* Разные кодировки подходят для разных моделей
* Масштабирование улучшает работу алгоритмов (например, SVM, KNN)

